# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Lane 2: Refresh / Content Opportunity Scoring.**

> **Which pages should a content editor review first for refresh, expansion, protection, pruning, or monitoring?**

**The decision this supports.** Not "predict decline", but the *order* of a weekly review queue. In this dataset 9,961 pages are both declining and still earning impressions; at 50 reviews a week that is roughly **199 weeks of backlog**. Nobody clears that list, so the only lever anyone actually has is which pages come first.

**Who acts, and how.** A content/SEO editor takes the top N pages their week allows and picks one action per page (refresh, expand, protect, prune, or monitor) with a reason code attached so they can disagree with the ranking.

**Why the two errors are not equal.** A false positive costs 1–3 editor hours and is visible. A false negative leaves a declining page with real traffic to keep sliding, and nobody ever learns it was missed. That asymmetry shapes both the metric and the human-review rules.

**Task type:** ranking/scoring. **Metric, fixed before training:** precision@50, one editor-week of queue, always reported beside its base rate.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [1]:
# DATA. Lane 2's documented default is "starter playground plus warehouse support"
# (docs/ml-intern-dataset-and-lane-guide.md) -- the model is built on the starter
# release; the warehouse supplied the data contract and window design in ML-04.
import os, json
from pathlib import Path
import numpy as np, pandas as pd, sklearn

SEED, VISIBLE_MIN, K = 42, 500, 50
if Path.cwd().name == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("RELEASE: content_refresh_anonymized.csv (starter playground)")
print(f"  rows (one row = one page)  : {len(df):,}")
print(f"  pseudonymised clients      : {df['client_id'].nunique()}")
print(f"  columns                    : {df.shape[1]}")
print(f"  window                     : trailing 90 days, one snapshot")
print(f"  label positive rate        : {df['is_declining_label'].mean():.3f}")
print(f"  grain holds (id unique)    : {df['content_id'].is_unique}")

print("\nWAREHOUSE SUPPORT (ML-04 data contract):")
print("  FlyRank/internship-warehouse, fact_content_daily_performance")
print("  month=2026-03 features -> month=2026-04 label, decision moment 2026-03-31")
print("  9,841,378 March page-days; only 36.7% carried GSC data (IS TRUE), 4.2% GA4")

print("\nEXCLUDED, AND WHY:")
for col, why in [
    ("trend_direction", "the label is this column bucketed"),
    ("trend_pct", "the label's own source; notebook 02 shows the leak"),
    ("impressions_last_30d / _prev_30d", "trend_pct IS their ratio - the pair reconstructs the label"),
    ("all GA4 columns (warehouse)", "present on 4.2% of rows; zero-filled elsewhere, so 0 means 'not measured'"),
    ("content_id / client_id", "pseudonyms - grouping and splitting only, never features"),
    ("pages under 500 impressions/90d", "below the visibility floor: no edit pays off on traffic nobody sees"),
]:
    print(f"  - {col:35s} {why}")

print("\nPUBLIC-SAFE: this release ships no client names, domains, URLs, page titles")
print("or raw queries. Nothing in this notebook prints an identifier that is not a hash.")


RELEASE: content_refresh_anonymized.csv (starter playground)
  rows (one row = one page)  : 30,000
  pseudonymised clients      : 32
  columns                    : 45
  window                     : trailing 90 days, one snapshot
  label positive rate        : 0.542
  grain holds (id unique)    : True

WAREHOUSE SUPPORT (ML-04 data contract):
  FlyRank/internship-warehouse, fact_content_daily_performance
  month=2026-03 features -> month=2026-04 label, decision moment 2026-03-31
  9,841,378 March page-days; only 36.7% carried GSC data (IS TRUE), 4.2% GA4

EXCLUDED, AND WHY:
  - trend_direction                     the label is this column bucketed
  - trend_pct                           the label's own source; notebook 02 shows the leak
  - impressions_last_30d / _prev_30d    trend_pct IS their ratio - the pair reconstructs the label
  - all GA4 columns (warehouse)         present on 4.2% of rows; zero-filled elsewhere, so 0 means 'not measured'
  - content_id / client_id              ps

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [2]:
# METHODOLOGY: features, label, baseline, validation design, leakage checks.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

for c in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{c}"] = np.log1p(df[c])                     # traffic is heavy-tailed

NUM = ["search_volume", "competition", "cpc", "word_count", "char_count",
       "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
       "days_with_impressions", "days_with_sessions", "content_age_days",
       "days_since_last_update", "ctr", "avg_position", "engagement_rate",
       "scroll_rate", "ai_traffic_pct"]
CAT = ["competition_level", "content_type", "main_intent", "age_tier",
       "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]

print("LABEL      is_declining_label = (trend_direction == 'down')")
print("           An observed impressions-trend bucket. NOT page quality, NOT the future.")
print(f"FEATURES   {len(NUM)} numeric + {len(CAT)} categorical, all pre-decision\n")

# --- LEAKAGE CHECK 1: assert the banned columns are absent -------------------
BANNED = {"trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
          "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
          "content_id", "client_id"}
leak = set(NUM + CAT) & BANNED
assert not leak, f"LEAK: {leak}"
print(f"LEAKAGE CHECK 1  banned columns in feature set: {sorted(leak) or 'NONE'}  OK")

def pipe(num_cols):
    return Pipeline([("prep", ColumnTransformer([
        ("n", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num_cols),
        ("c", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                        ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), CAT)])),
        ("m", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                     class_weight="balanced", n_jobs=-1, random_state=SEED))])

def precision_at_k(scores, labels, k=K):
    return float(np.asarray(labels)[np.argsort(-np.asarray(scores))[:k]].mean())

y = df["is_declining_label"].values

# --- LEAKAGE CHECK 2: positive control. A test that never fires proves nothing.
g_tr, g_te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
                  .split(df, y, groups=df["client_id"]))
honest = roc_auc_score(y[g_te], pipe(NUM).fit(df.iloc[g_tr][NUM + CAT], y[g_tr])
                       .predict_proba(df.iloc[g_te][NUM + CAT])[:, 1])
leaked = roc_auc_score(y[g_te], pipe(NUM + ["trend_pct"]).fit(df.iloc[g_tr][NUM + ["trend_pct"] + CAT], y[g_tr])
                       .predict_proba(df.iloc[g_te][NUM + ["trend_pct"] + CAT])[:, 1])
print(f"LEAKAGE CHECK 2  positive control: ROC-AUC {honest:.3f} honest -> {leaked:.3f} with trend_pct")
print("                 the harness detects a leak when one exists, so the honest number is")
print("                 honest because the features are clean, not because the test is blind.")

# --- VALIDATION DESIGN ------------------------------------------------------
print(f"\nVALIDATION  GroupShuffleSplit on client_id, 25% of CLIENTS held out, 8 repeats.")
print("            Grouped because pages within a client share templates and cadence:")
print("            a random row split lets the model recognise the client instead of the")
print("            pattern. Measured cost of getting this wrong: +0.157 precision@50 of")
print("            illusion (ML-09). Repeated because one split moved results by up to 0.14.")

# --- BASELINE (frozen at ML-07) ---------------------------------------------
def baseline_rule(train, test):
    """Transparent hand rule. Band medians fitted on TRAIN only."""
    def prep(d):
        d = d.copy()
        d["visible"] = d["impressions_90d"] >= VISIBLE_MIN
        d["has_pos"] = d["avg_position"] > 0             # 0 = no data, not rank zero
        d["top20"] = d["has_pos"] & (d["avg_position"] <= 20)
        d["pos_band"] = pd.cut(d["avg_position"].where(d["has_pos"]),
                               [0, 10, 20, 1000], labels=["1-10", "11-20", "21+"])
        return d
    tr, te = prep(train), prep(test)
    bm = te["pos_band"].map(tr.groupby("pos_band", observed=True)["ctr"].median()).astype(float)
    shortfall = (bm - te["ctr"]).clip(lower=0).fillna(0)
    w = np.where(~te["visible"], 0,
        np.where(te["top20"] & (te["ctr"] < bm), 3,
        np.where(te["freshness_tier"].eq("91-180"), 2, 1)))
    return w * np.log10(te["impressions_90d"].clip(lower=1)) * (1 + shortfall)

print("\nBASELINE    score = weight x log10(impressions) x (1 + CTR shortfall), where weight")
print("            comes from one reason code per page. Frozen at ML-07 and never retuned.")


LABEL      is_declining_label = (trend_direction == 'down')
           An observed impressions-trend bucket. NOT page quality, NOT the future.
FEATURES   18 numeric + 8 categorical, all pre-decision

LEAKAGE CHECK 1  banned columns in feature set: NONE  OK


LEAKAGE CHECK 2  positive control: ROC-AUC 0.730 honest -> 1.000 with trend_pct
                 the harness detects a leak when one exists, so the honest number is
                 honest because the features are clean, not because the test is blind.

VALIDATION  GroupShuffleSplit on client_id, 25% of CLIENTS held out, 8 repeats.
            Grouped because pages within a client share templates and cadence:
            a random row split lets the model recognise the client instead of the
            pattern. Measured cost of getting this wrong: +0.157 precision@50 of
            illusion (ML-09). Repeated because one split moved results by up to 0.14.

BASELINE    score = weight x log10(impressions) x (1 + CTR shortfall), where weight
            comes from one reason code per page. Frozen at ML-07 and never retuned.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [3]:
# RESULTS: model vs baseline, same data, same 8 client-grouped splits, same metric.
MODELS = {
    "logistic_regression": lambda: Pipeline(pipe(NUM).steps[:-1] + [("m", LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=SEED))]),
    "decision_tree_d3": lambda: Pipeline(pipe(NUM).steps[:-1] + [("m", DecisionTreeClassifier(
        max_depth=3, class_weight="balanced", random_state=SEED))]),
    "random_forest": lambda: pipe(NUM),
    "gradient_boosting": lambda: Pipeline(pipe(NUM).steps[:-1] + [("m", GradientBoostingClassifier(
        random_state=SEED))]),
}

rows = []
for seed in range(8):
    tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
                      .split(df, y, groups=df["client_id"]))
    train, test, yte = df.iloc[tr_i], df.iloc[te_i], y[te_i]
    rows.append(dict(seed=seed, model="base rate (floor)", p50=yte.mean(), auc=0.5))
    bs = baseline_rule(train, test)
    rows.append(dict(seed=seed, model="baseline rule (hand-written)",
                     p50=precision_at_k(bs, yte), auc=roc_auc_score(yte, bs)))
    for name, make in MODELS.items():
        prob = make().fit(train[NUM + CAT], y[tr_i]).predict_proba(test[NUM + CAT])[:, 1]
        rows.append(dict(seed=seed, model=name, p50=precision_at_k(prob, yte),
                         auc=roc_auc_score(yte, prob)))

res = pd.DataFrame(rows)
ORDER = ["base rate (floor)", "baseline rule (hand-written)", "logistic_regression",
         "decision_tree_d3", "random_forest", "gradient_boosting"]
TABLE = (res.groupby("model").agg(**{"precision@50": ("p50", "mean"),
                                     "sd": ("p50", "std"), "ROC-AUC": ("auc", "mean")})
         .reindex(ORDER).round(3))
print("=" * 74)
print("MODEL vs BASELINE - same data, same 8 client-grouped splits, same metric")
print("=" * 74)
print(TABLE.to_string())

# Paired: every method saw the SAME splits, so compare split-by-split.
P = res.pivot(index="seed", columns="model", values="p50")
print("\nPAIRED comparison (how often did it actually win, not just on average):")
PAIRS = {}
for a, b in [("random_forest", "baseline rule (hand-written)"),
             ("logistic_regression", "baseline rule (hand-written)"),
             ("random_forest", "gradient_boosting")]:
    d = P[a] - P[b]
    PAIRS[f"{a} vs {b}"] = {"mean": round(float(d.mean()), 3), "wins": int((d > 0).sum()),
                            "of": 8, "worst": round(float(d.min()), 3)}
    print(f"  {a:<20} vs {b:<28} {d.mean():+.3f}  wins {int((d>0).sum())}/8  worst {d.min():+.3f}")

print("\nREADING IT: the forest beats the hand rule in 7 of 8 splits (+0.200) - a real gain.")
print("Gradient boosting beats the forest in 5 of 8 (+0.022) - a coin flip, so the extra")
print("complexity does not earn its place and the simpler model ships.")
print("Split 3 is the honest counter-example: there the hand rule scored 0.80 and the")
print("forest 0.60. That is what a 7-of-8 win rate looks like from the inside.")


MODEL vs BASELINE - same data, same 8 client-grouped splits, same metric
                              precision@50     sd  ROC-AUC
model                                                     
base rate (floor)                    0.495  0.068    0.500
baseline rule (hand-written)         0.605  0.114    0.583
logistic_regression                  0.735  0.120    0.673
decision_tree_d3                     0.622  0.126    0.657
random_forest                        0.805  0.108    0.710
gradient_boosting                    0.782  0.146    0.712

PAIRED comparison (how often did it actually win, not just on average):
  random_forest        vs baseline rule (hand-written) +0.200  wins 7/8  worst -0.200
  logistic_regression  vs baseline rule (hand-written) +0.130  wins 7/8  worst -0.080
  random_forest        vs gradient_boosting            +0.022  wins 5/8  worst -0.140

READING IT: the forest beats the hand rule in 7 of 8 splits (+0.200) - a real gain.
Gradient boosting beats the forest in 5

## 5. Limitations

*What this work cannot claim.*

Written before a reader can write it for me.

**1. No causal claim is available.** Nobody refreshed a random half of these pages. I observed that declining pages share certain properties; I cannot say that refreshing them recovers traffic. The freshness pattern in §6 is an association in one snapshot, and I do not build a scheduling policy on it.

**2. The label is a trend bucket, not the future and not quality.** `is_declining_label` comes from `trend_direction`, an impressions-trend bucket already computed in the release. A declining page is not a bad page. The warehouse work in ML-04 built a genuinely forward-looking label (March features → April outcome) and showed the framing survives; the model here still rests on the snapshot version.

**3. The model is weakest exactly where it matters most.** Accuracy is 0.569 on the largest (`excellent`) impression tier against 0.725 on the smallest, and the top-50 queue caught **0 of 259** declining pages in that tier. High-traffic pages, the ones a client asks about first, need a human-led process, not this queue.

**4. It cannot tell "just arrived" from "on the way out".** The most confident false positives were pages with impressions and zero clicks whose real trend was *up*. One 90-day snapshot has no history, so a rising page and a dying page look alike.

**5. Small panel, wide spreads.** 32 pseudonymised clients, 8 held out per split. Precision@50 moves ±0.11 between splits, so I quote "roughly 0.80, ±0.11", never 0.805. Differences smaller than that spread are not differences. One held-out client scored 0.445, below a coin flip.

**6. One portfolio, one quarter.** Nothing here establishes that the ordering transfers to a different client mix or a different period.

**7. A confounder I did not fully resolve.** Page age could drive both staleness and decline. Permutation importance ranks `content_age_days` third and shuffling it costs only ~0.021 AUC, so the model is not merely an age detector. But that measures what the model leans on, not whether age confounds staleness. That needs a stratified comparison I did not run.

**Claim language used throughout:** *observed* for patterns in this dataset, *directional* for comparisons that survived repetition, *decision-support* for the queue. Never *proves*, *causes*, *will increase*, or any statement about Google's algorithm.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [4]:
# RANKED RECOMMENDATIONS: the queue the paper publishes.
# Every page scored OUT-OF-FOLD, so no page is ranked by a model that saw its own client.
df["risk_score"] = cross_val_predict(pipe(NUM), df[NUM + CAT], y, cv=GroupKFold(n_splits=5),
                                     groups=df["client_id"], method="predict_proba")[:, 1]

df["visible"] = df["impressions_90d"] >= VISIBLE_MIN
df["has_position"] = df["avg_position"] > 0
df["pos_band"] = pd.cut(df["avg_position"].where(df["has_position"]),
                        [0, 10, 20, 1000], labels=["1-10", "11-20", "21+"])
df["ctr_below_band"] = df["ctr"] < df.groupby("pos_band", observed=True)["ctr"].transform("median")

def classify(r):
    if not r.visible:
        return ("low_visibility", "NO ACTION", "below the 500 impressions/90d visibility floor")
    if r.has_position and r.avg_position <= 20 and r.ctr_below_band:
        return ("page_one_underperformer", "FIX CTR", "ranks 1-20 but CTR sits below its position band")
    if r.freshness_tier == "91-180":
        return ("stale_but_visible", "REFRESH", "not updated in 91-180 days and still earning impressions")
    if r.has_position and 10 < r.avg_position <= 20:
        return ("striking_distance", "SUPPORT", "just outside page 1 with competitive CTR")
    if r.has_position and r.avg_position > 20:
        return ("deep_and_visible", "MONITOR", "ranks past position 20 - CTR logic does not apply here")
    return ("visible_no_flag", "MONITOR", "visible, but no archetype flag fired")

df[["archetype", "action", "reason_code"]] = df.apply(classify, axis=1, result_type="expand")
queue = df[df["visible"]].sort_values("risk_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
QUEUE_BASE = queue["is_declining_label"].mean()
QUEUE_P50 = precision_at_k(queue["risk_score"], queue["is_declining_label"])

print("THE MODEL RANKS. A READABLE RULE NAMES THE ACTION. A PERSON DECIDES.\n")
print(f"queue: {len(queue):,} reviewable pages | in-queue base rate {QUEUE_BASE:.3f} "
      f"| out-of-fold precision@50 {QUEUE_P50:.3f}\n")
ARCH = (queue.groupby(["archetype", "action"], observed=True)
        .agg(pages=("rank", "size"), observed_decline_rate=("is_declining_label", "mean"))
        .sort_values("pages", ascending=False).round(3))
print("ARCHETYPE -> ACTION"); print(ARCH.to_string())

FRESH = (df.groupby("freshness_tier", observed=True)["is_declining_label"]
         .agg(n="size", decline_rate="mean").reindex(["0-30", "31-90", "91-180", "181+"]))
FRESH["n"] = FRESH["n"].astype(int)
print(f"\nDECAY / REFRESH SIGNAL (portfolio base rate {df['is_declining_label'].mean():.3f})")
print(FRESH.round(3).to_string())
print("Decline peaks at 91-180 days since last update. The 31-90 and 181+ tiers carry")
print("n=175 and n=174 and the stalest tier REVERSES, so only the 91-180 window is acted on.")

GAIN = (QUEUE_P50 - QUEUE_BASE) * K
print(f"\nVALUE: at {K} reviews/week the queue surfaces ~{QUEUE_P50*K:.0f} genuinely declining")
print(f"pages vs ~{QUEUE_BASE*K:.0f} in random order = ~{GAIN:.0f} better-targeted reviews per editor-week.")
print("A meaningful scheduling improvement, not a transformation.\n")
print("NOT AUTOMATED: content edits, pruning/deletion, client-facing forecasts,")
print("high-traffic prioritisation, performance management, scoring new clients unreviewed,")
print("auto-refresh scheduling. The honest gain does not justify removing the human.")


THE MODEL RANKS. A READABLE RULE NAMES THE ACTION. A PERSON DECIDES.

queue: 16,726 reviewable pages | in-queue base rate 0.596 | out-of-fold precision@50 0.840

ARCHETYPE -> ACTION
                                 pages  observed_decline_rate
archetype               action                               
stale_but_visible       REFRESH   5081                  0.579
page_one_underperformer FIX CTR   3673                  0.701
visible_no_flag         MONITOR   3447                  0.531
deep_and_visible        MONITOR   2485                  0.571
striking_distance       SUPPORT   2040                  0.584

DECAY / REFRESH SIGNAL (portfolio base rate 0.542)
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471
Decline peaks at 91-180 days since last update. The 31-90 and 181+ tiers carry
n=175 and n=174 and the stalest tier 

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [5]:
# ARTIFACTS THE PAPER EMBEDS. Written into docs/ so the deployed folder is
# self-contained and every asset path in index.html is RELATIVE.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DOCS = Path("docs/paper")   # own subfolder: docs/ already holds the repo docs
(DOCS / "img").mkdir(parents=True, exist_ok=True)

# Palette validated with the dataviz checker on the light surface (all six checks pass).
BLUE, ORANGE, SURFACE = "#2a78d6", "#eb6834", "#fcfcfb"
PALE, INK, MUTED = "#b9d3f2", "#0b0b0b", "#52514e"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "font.size": 10,
                     "axes.edgecolor": "#d8d7d2", "text.color": INK, "axes.labelcolor": MUTED,
                     "xtick.color": MUTED, "ytick.color": MUTED})

def strip(ax):
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.tick_params(length=0)

# FIGURE 1 -- the headline result, with its spread shown rather than hidden.
labels = ["Base rate\n(random order)", "Hand rule\n(baseline)", "Logistic\nregression",
          "Random forest\n(shipped)"]
keys = ["base rate (floor)", "baseline rule (hand-written)", "logistic_regression", "random_forest"]
vals = [TABLE.loc[k, "precision@50"] for k in keys]
sds = [0.0 if pd.isna(TABLE.loc[k, "sd"]) else TABLE.loc[k, "sd"] for k in keys]

fig, ax = plt.subplots(figsize=(7.6, 4.2))
ax.bar(labels, vals, width=0.62, color=[MUTED, ORANGE, BLUE, BLUE], zorder=3,
       edgecolor=SURFACE, linewidth=2)
ax.errorbar(labels, vals, yerr=sds, fmt="none", ecolor=MUTED, elinewidth=1.2, capsize=5, zorder=4)
for i, v in enumerate(vals):
    ax.text(i, v - 0.06, f"{v:.2f}", ha="center", fontsize=12, fontweight="bold", color="#ffffff")
ax.axhline(vals[0], color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)
ax.set_ylim(0, 1.0); ax.set_ylabel("Precision@50 (held-out clients)")
ax.set_title("Ranking a review queue, measured against the floor",
             fontsize=12.5, fontweight="bold", loc="left", pad=12)
ax.yaxis.grid(True, color="#eceae4", lw=1, zorder=0); strip(ax)
fig.text(0.01, -0.03, "8 client-grouped splits; error bars = 1 sd. Bars whose error bars "
                      "overlap are not distinguishable at this sample size.", fontsize=8, color=MUTED)
fig.tight_layout(); fig.savefig(DOCS / "img/fig1_results.png", dpi=200,
                                bbox_inches="tight", facecolor=SURFACE)
plt.close(fig)

# FIGURE 2 -- the decay signal, with sample size encoded in the mark itself.
small = FRESH["n"] < 1000
fig, ax = plt.subplots(figsize=(7.6, 4.2))
ax.bar(FRESH.index, FRESH["decline_rate"], width=0.62, zorder=3, edgecolor=SURFACE, linewidth=2,
       color=[(PALE if s else BLUE) for s in small], hatch=["///" if s else "" for s in small])
for i, (_, row) in enumerate(FRESH.iterrows()):
    on_bar = INK if small.iloc[i] else "#ffffff"
    ax.text(i, row["decline_rate"] - 0.045, f"{row['decline_rate']:.3f}",
            ha="center", fontsize=12, fontweight="bold", color=on_bar)
    ax.text(i, 0.022, f"n = {int(row['n']):,}", ha="center", fontsize=9, color=on_bar)
BASE_RATE = df["is_declining_label"].mean()
ax.axhline(BASE_RATE, color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)
ax.text(-0.44, BASE_RATE + 0.012, f"portfolio base rate {BASE_RATE:.3f}",
        ha="left", fontsize=8.5, color=MUTED)
ax.set_ylim(0, 0.72); ax.set_ylabel("Share of pages observed declining")
ax.set_xlabel("Days since last update")
ax.set_title("Decline peaks 91-180 days after the last update",
             fontsize=12.5, fontweight="bold", loc="left", pad=12)
ax.yaxis.grid(True, color="#eceae4", lw=1, zorder=0); strip(ax)
fig.text(0.01, -0.03, "Hatched bars carry n < 1,000 and are not reliable -- the stalest tier "
                      "reverses. Observed association in one 90-day snapshot; not evidence that "
                      "refreshing causes recovery.", fontsize=8, color=MUTED)
fig.tight_layout(); fig.savefig(DOCS / "img/fig2_decay.png", dpi=200,
                                bbox_inches="tight", facecolor=SURFACE)
plt.close(fig)

# The numbers the paper quotes, so every figure on the page traces back to a run.
CAPSTONE = {
    "seed": SEED, "sklearn_version": sklearn.__version__, "splits": 8,
    "split_design": "GroupShuffleSplit on client_id, 25% of clients held out",
    "portfolio": {"pages": int(len(df)), "clients": int(df["client_id"].nunique()),
                  "base_rate": round(float(BASE_RATE), 4)},
    "results_table": TABLE.to_dict(orient="index"),
    "paired": PAIRS,
    "queue": {"rows": int(len(queue)), "base_rate": round(float(QUEUE_BASE), 4),
              "out_of_fold_precision_at_50": round(float(QUEUE_P50), 4),
              "net_gain_pages_per_week": round(float(GAIN), 1)},
    "archetypes": {a: {"action": g["action"].iloc[0], "pages": int(len(g))}
                   for a, g in queue.groupby("archetype", observed=True)},
    "decay_by_freshness": {k: {"n": int(v["n"]), "decline_rate": round(float(v["decline_rate"]), 4)}
                           for k, v in FRESH.iterrows()},
    "leakage_positive_control": {"honest_auc": round(float(honest), 4),
                                 "with_trend_pct_auc": round(float(leaked), 4)},
}
(DOCS / "capstone_metrics.json").write_text(json.dumps(CAPSTONE, indent=2, default=float))

print("ARTIFACTS WRITTEN (all paths relative, ready for GitHub Pages):")
for p in sorted(DOCS.rglob("*")):
    if p.is_file():
        print(f"  {p}")


ARTIFACTS WRITTEN (all paths relative, ready for GitHub Pages):
  docs\paper\capstone_metrics.json
  docs\paper\img\fig1_results.png
  docs\paper\img\fig2_decay.png


## 8. Five-minute demo outline

*Week-8 showcase. The timings are the point. This runs in five minutes or it is not an outline.*

**0:00 to 0:45 | The question**

One line on the screen: **which pages should a content editor open first?** In this dataset 9,961
pages are losing impressions *and* still earning real search visibility. At a realistic 50 reviews a
week that is 199 weeks of backlog. Nobody clears that list, so the only lever anyone has is the
order. FlyRank already flags these pages with hand-written rules, but a flag answers *is this page
in trouble*, not *which one do I open on Monday*.

**0:45 to 2:00 | The method**

- **Label:** the page is trending down over the 90-day window. This is a ranking problem, so the
  metric is precision@50, one editor-week of queue, always shown next to the base rate.
- **Baseline:** the hand-written rule, reimplemented transparently in `w04`. Not a strawman. It is
  the thing already running in production.
- **Validation:** splits grouped by client, 8 of them. No model is ever scored on a client it
  trained on.
- **Product flags are outputs, never features.** Health score and quick-win tags encode a decision
  someone already made, so using them would make the result circular.
- **Leakage checks run in both directions.** I deliberately fed the harness a leaked feature to
  confirm it would catch one.

**2:00 to 3:15 | One chart**

`docs/paper/img/fig1_results.png`, precision@50 for base rate, hand rule, logistic, tree, random
forest and gradient boosting, with the base-rate reference line drawn behind the bars. Say the
takeaway out loud rather than reading the axes: **0.49 base, 0.61 rule, 0.80 forest.**

**3:15 to 4:15 | One honest result**

The random forest reaches **precision@50 of 0.80, plus or minus 0.11**, and beats the rule in **7 of
8 splits**. The honest part is that spread and the split I lost. The same model on *random row*
splits reads 0.96, and that number is not real: it comes from the same client sitting on both sides
of the split. Grouping by client cost 0.157 and bought the number its credibility. Gradient boosting
won only 5 of 8 against the forest, so I did not adopt it.

**4:15 to 5:00 | One recommendation**

Work the top 50 of the out-of-fold queue each week. Precision@50 is 0.84 there against a 0.60 base
rate inside the queue, so roughly **12 more genuinely declining pages per editor-week**. Every row
carries an archetype and a reason code, so the editor knows why it surfaced. It is a reading order,
not a verdict, and nothing here shows that refreshing a page recovers its traffic.

**Expect this question:** *would it hold on production data?* Honest answer: unknown. One 90-day
snapshot, 32 pseudonymised clients, one label definition. Directional, not proven.


## 9. Two shareable cuts

*The same work, retold for two audiences. Both are public-safe: no client names, domains, URLs or
queries.*

### Social post, the methodology cut

> My first model scored 0.96. I threw it away.
>
> The task: rank which web pages a content editor should review first, on an anonymised 30,000-page
> search dataset covering 32 pseudonymised client sites.
>
> Splitting rows at random let the *same client* land on both sides of the split, so the model was
> partly recognising clients rather than decay. Splitting by client instead dropped precision@50
> from 0.96 to 0.80. That missing 0.16 was never real. I just could not see it.
>
> Then I checked the harness could still catch me. I deliberately fed it a feature built from the
> label window and it scored a perfect 1.000. Good. That is exactly what a leak should look like
> when your validation is working.
>
> Final, on identical client-grouped splits: **0.80** for the model, **0.61** for the hand-written
> rule it has to beat, **0.49** base rate.
>
> Write-up and notebooks: https://abbad0708.github.io/FlyRankAI-ML-Internship/paper/
> Data: the FlyRank ML Internship dataset (flyrank.ai).

### Employer summary, three sentences

> I built a ranked review queue that tells a content editor which pages to open first, trained on an
> anonymised 30,000-page search dataset covering 32 pseudonymised client sites, using real Search
> Console and Analytics data rather than a toy set.
>
> It reaches precision@50 of about 0.80 against 0.61 for the hand-written rule it has to beat and a
> 0.49 base rate, measured on client-grouped splits so no model is ever scored on a client it
> trained on.
>
> In practice that surfaces roughly 12 more genuinely declining pages per editor-week: decision
> support with a stated reason on every row, not a claim that refreshing a page recovers its
> traffic.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — hashed identifiers only
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Still to do by hand (they cannot be done from a notebook)

- [ ] Create the GitHub repo from the template and push this work
- [ ] Repo → Settings → Pages → Deploy from branch `main`, folder `/docs`
- [ ] Open the live URL in an incognito window **and** on a phone; confirm both figures render
- [ ] Paste that exact URL into `submission/paper_url.txt` — one line, nothing else
- [ ] Submit the repo URL on the ML-11 card and on the capstone submission panel

**Reproducibility.** Seed 42 throughout; 8 fixed client splits (0–7); library versions recorded in `docs/paper/capstone_metrics.json` alongside the results. Tree-ensemble numbers move a few points between scikit-learn versions, so a rerun should reproduce the ordering and the win counts, not necessarily the third decimal.
